**IMPORTANT: CLEAR OUTPUTS BEFORE PUSHING**

In [ ]:
import pandas as pd

PRE_SURVEY_FILEPATH = "input/pre_survey.csv"
DAILY_SURVEY_FILEPATH = "input/daily_survey.csv"

pre_survey_df = pd.read_csv(PRE_SURVEY_FILEPATH)
pre_survey_df = pre_survey_df.rename(columns={
    'tone condition': 'tone_condition',
    'topic condition': 'topic_condition',
    'pre-survey completion time': 'presurvey_timestamp',
    'registration date': 'registration_date'
})

daily_survey_df = pd.read_csv(DAILY_SURVEY_FILEPATH)
daily_survey_df.columns = daily_survey_df.columns.str.split('.').str[0]
daily_survey_df = daily_survey_df.rename(columns={
    'Email Address': 'email',
    """Did the bot respond to your mention with a fact-check? If the bot responded that the claim was not able to be fact-checked, or if you accidentally missed one of your 10 posts from the previous day, select "No""": 'valid_reply',
    """Four-Letter Code""": 'reply_id',
    """To confirm that you viewed the bot's reply, write the username of the user who posted the original post that you replied to when mentioning the bot""": 'parent_author',
    """Rate your agreement with the following statement: I enjoyed this interaction with the bot""": 'enjoyment',
    """Rate your agreement with the following statement: The bot's post was informative""": 'informativeness',
    """Rate your agreement with the following statement: I was satisfied with the quality of the bot's fact-check""": 'satisfaction',
    """(Optional) If any specific details about the bot's reply informed your answers to the questions above, please provide a 2-3 sentence reasoning here""": 'long_answer_main',
    """(If applicable) Rate your agreement with the following statement: Reading the replies to the bot's post makes me interested in following the discussion thread on this post""": 'reply_interest_following',
    """(If applicable)  Rate your agreement with the following statement: Reading the replies to the bot's post makes me interested in engaging in the discussion thread on this post""": 'reply_interest_engaging',
    """(If applicable) Please provide a 2-3 sentence reasoning for your answers to the questions above""": 'long_answer_extra'
})

In [ ]:
pre_survey_df.head(10)

In [ ]:
daily_survey_df.head(10)

In [ ]:
for i in range(2, len(daily_survey_df.columns.values)):
    daily_survey_df.columns.values[i] = f"{daily_survey_df.columns.values[i]}{int((i - 1) / 10)}"
daily_survey_df.head(10)
daily_survey_df_long = pd.wide_to_long(
    daily_survey_df,
    stubnames=[
        'valid_reply',
        'reply_id',
        'parent_author',
        'enjoyment',
        'informativeness',
        'satisfaction',
        'long_answer_main',
        'reply_interest_following',
        'reply_interest_engaging',
        'long_answer_extra',
    ],
    i='Timestamp',
    j='temp',
).reset_index()[[
    'Timestamp',
    'email',
    'valid_reply',
    'reply_id',
    'parent_author',
    'enjoyment',
    'informativeness',
    'satisfaction',
    'long_answer_main',
    'reply_interest_following',
    'reply_interest_engaging',
    'long_answer_extra',
]]
daily_survey_df_long.head(10)

In [ ]:
daily_survey_df_full = pd.merge(
    daily_survey_df_long,
    pre_survey_df,
    on='email',
    how='left'
)
daily_survey_df_full.head(10)

In [ ]:
daily_survey_df_final = daily_survey_df_full[(daily_survey_df_full['valid_reply'] == 'Yes')]
daily_survey_df_final = daily_survey_df_final.dropna(subset=[
    'email',
    'enjoyment',
    'informativeness',
    'satisfaction'
])
len(daily_survey_df_final)

In [ ]:
daily_survey_df_final.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')
style = {
    'boxprops': {'color': 'tab:blue'},
    'whiskerprops': {'color': 'tab:blue'},
    'medianprops': {'color': 'tab:green'}
}
enjoyment_boxplot = daily_survey_df_full.boxplot(column='enjoyment', by='tone_condition', **style)
plt.savefig("enjoyment_boxplot.png", bbox_inches='tight', dpi=300)
informativeness_boxplot = daily_survey_df_full.boxplot(column='informativeness', by='tone_condition', **style)
plt.savefig("informativeness_boxplot.png", bbox_inches='tight', dpi=300)
satisfaction_boxplot = daily_survey_df_full.boxplot(column='satisfaction', by='tone_condition', **style)
plt.savefig("satisfaction_boxplot.png", bbox_inches='tight', dpi=300)

In [ ]:
import statsmodels.formula.api as smf

pvals = {}
def record_pvals(results, reference_group, dependent_var):
    if dependent_var not in pvals:
        pvals[dependent_var] = {}
    for treatment_group in ['agreeable', 'neutral', 'satirical']:
        if treatment_group != reference_group:
            # Generate standard ordering for each group pair to avoid adding duplicates
            comparison_groups = [reference_group, treatment_group]
            comparison_groups.sort()
            pval_key = f'{comparison_groups[0]}-{comparison_groups[1]}'

            # Add pval to dict
            if pval_key not in pvals[dependent_var]:
                results_key = f"C(tone_condition, Treatment(reference='{reference_group}'))[T.{treatment_group}]"
                pvals[dependent_var][pval_key] = results.pvalues[results_key]

def model(reference_group, dependent_var, data=None):
    if data is None:
        data = daily_survey_df_final

    model_intercept = smf.mixedlm(
        f"{dependent_var} ~ C(tone_condition, Treatment(reference='{reference_group}'))",
        data,
        groups=data['email']
    )
    result_intercept = model_intercept.fit()

    print()
    print(f"Reference Group: {reference_group} | Dependent Variable: {dependent_var}")
    print(result_intercept.summary())

    record_pvals(result_intercept, reference_group, dependent_var)
    return result_intercept


def model_all():
    for reference_group in ['agreeable', 'neutral', 'satirical']:
        for dependent_var in ['enjoyment', 'informativeness', 'satisfaction']:
            model(reference_group=reference_group, dependent_var=dependent_var)

model_all()
print(pvals)

In [ ]:
import json
import statsmodels.stats.multitest as multitest

raw_pvals = []
for comparison in ['agreeable-satirical', 'neutral-satirical', 'agreeable-satirical']:
    for dependent_var in ['enjoyment', 'informativeness', 'satisfaction']:
        raw_pvals.append(pvals[dependent_var][comparison])

_, raw_pvals_corrected, _, _ = multitest.multipletests(raw_pvals, alpha=0.05, method='holm')
pvals_corrected = {}
i = 0
for comparison in ['agreeable-satirical', 'neutral-satirical', 'agreeable-satirical']:
    for dependent_var in ['enjoyment', 'informativeness', 'satisfaction']:
        if dependent_var not in pvals_corrected:
            pvals_corrected[dependent_var] = {}
        pvals_corrected[dependent_var][comparison] = raw_pvals[i]
        i += 1

print(json.dumps(pvals_corrected, indent=4))